In [1]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import plotly.express as px


def find_models(base_path):
    return sorted(
        p.parent.parent.name
        for p in Path(base_path).glob("*/promptfoo-db/promptfoo.db")
    )


def load_model(base_path, model):
    db_path = Path(base_path) / model / "promptfoo-db" / "promptfoo.db"

    query = """
        SELECT id, test_idx, score, success, named_scores
        FROM eval_results
    """

    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(query, conn)

    scores = df["named_scores"].apply(lambda x: json.loads(x) if x else {})
    scores = pd.json_normalize(scores)

    df = pd.concat(
        [df.drop(columns=["named_scores"]), scores],
        axis=1,
    )

    df["model"] = model
    df.columns = [c.replace("-", "_") for c in df.columns]

    return df


def load_models(base_path, models=None):
    if models is None:
        models = find_models(base_path)

    return pd.concat(
        [load_model(base_path, model) for model in models],
        ignore_index=True,
    )


def to_long(df):
    meta_cols = ["model", "id", "test_idx", "score", "success"]
    metric_cols = [c for c in df.columns if c not in meta_cols]

    df_long = df.melt(
        id_vars=meta_cols,
        value_vars=metric_cols,
        var_name="metric",
        value_name="value",
    )

    df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
    return df_long.dropna(subset=["value"])


def plot_metrics(df_long, metrics=None, title="Promptfoo Metrics by Model"):
    if metrics is not None:
        df_long = df_long[df_long["metric"].isin(metrics)]

    fig = px.box(
        df_long,
        x="model",
        y="value",
        color="model",
        points="all",
        facet_col="metric",
        facet_col_wrap=4,
        title=title,
    )

    fig.update_xaxes(matches=None)
    fig.update_yaxes(matches=None)
    fig.update_layout(height=1400, showlegend=False)
    fig.show()

In [2]:
base_path = "/home/jonas/SynologyDrive/Uni-MA/2026_SoSe/Praktikum/results"

df = load_models(base_path)
df_long = to_long(df)

length_metrics = {"output_length", "reference_length"}

plot_metrics(
    df_long[~df_long["metric"].isin(length_metrics)],
    title="Promptfoo Score Metrics by Model",
)

plot_metrics(
    df_long[df_long["metric"].isin(length_metrics)],
    title="Promptfoo Length Metrics by Model",
)

In [8]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


def parse_json(x):
    try:
        return json.loads(x) if x else None
    except Exception:
        return x


def load_samples(base_path, model, n=20, best=False):
    db_path = Path(base_path) / model / "promptfoo-db" / "promptfoo.db"
    order = "DESC" if best else "ASC"

    query = f"""
        SELECT *
        FROM eval_results
        WHERE crystalbleu IS NOT NULL
        ORDER BY crystalbleu {order}
        LIMIT ?
    """

    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(query, conn, params=(n,))

    for col in ["test_case", "prompt", "response", "grading_result", "named_scores", "metadata"]:
        if col in df.columns:
            df[col] = df[col].apply(parse_json)

    df["model"] = model
    return df


def get_output(response):
    if isinstance(response, dict):
        return response.get("output") or response.get("tikz") or response.get("raw")
    return response


def make_image_df(samples, image_base_path, base_path, model):
    image_base_path = Path(image_base_path)
    generated_base_path = Path(base_path) / model / "generated_images"

    rows = []

    for rank, row in samples.reset_index(drop=True).iterrows():
        reference_img = Path(row["test_case"]["vars"]["reference_image"])
        reference_img = image_base_path / reference_img.name

        generated_img = generated_base_path / f"{reference_img.stem}_generated.png"

        rows.append({
            "rank": rank + 1,
            "test_idx": row["test_idx"],
            "score": row["score"],
            "reference_img": reference_img,
            "generated_img": generated_img if generated_img.exists() else None,
            "output": get_output(row["response"]),
        })

    return pd.DataFrame(rows)


def plot_image_matrix(img_df, n_cols=4):
    n_samples = len(img_df)
    n_rows = (n_samples + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows * 2,
        n_cols,
        figsize=(4 * n_cols, 6 * n_rows),
    )

    axes = axes.reshape(n_rows * 2, n_cols)

    for idx, (_, row) in enumerate(img_df.iterrows()):
        r = (idx // n_cols) * 2
        c = idx % n_cols

        # Reference oben
        ax_ref = axes[r, c]
        ax_ref.imshow(Image.open(row["reference_img"]))
        ax_ref.axis("off")
        ax_ref.set_title(
            f"rank {row['rank']} | test {row['test_idx']}\nscore: {row['score']}\nreference",
            fontsize=9,
        )

        # Generated darunter
        ax_gen = axes[r + 1, c]
        ax_gen.axis("off")

        if row["generated_img"] is not None:
            ax_gen.imshow(Image.open(row["generated_img"]))
            ax_gen.set_title("generated", fontsize=9)
        else:
            ax_gen.text(
                0.5,
                0.5,
                "no generated image",
                ha="center",
                va="center",
                fontsize=9,
            )

    # leere Achsen ausblenden
    used_cols_last_row = n_samples % n_cols or n_cols

    for idx in range(n_samples, n_rows * n_cols):
        r = (idx // n_cols) * 2
        c = idx % n_cols
        axes[r, c].axis("off")
        axes[r + 1, c].axis("off")

    plt.tight_layout()
    plt.show()


def show_samples(base_path, image_base_path, model, n=20, best=False, n_cols=5):
    samples = load_samples(base_path, model, n=n, best=best)
    img_df = make_image_df(samples, image_base_path, base_path, model)
    plot_image_matrix(img_df, n_cols=n_cols)
    return samples, img_df

In [9]:
base_path = "/home/jonas/SynologyDrive/Uni-MA/2026_SoSe/Praktikum/results"
image_base_path = "/home/jonas/Datasets/TikZ/our_dataset_benchmark_merged/simple_vlm_description/images"
model = "result-qwen3.6-35b-a3b"   #   geotikz-instruct   detikzify_2_5_8b

worst, worst_img_df = show_samples(
    base_path=base_path,
    image_base_path=image_base_path,
    model=model,
    n=20,
    best=False,
)

best, best_img_df = show_samples(
    base_path=base_path,
    image_base_path=image_base_path,
    model=model,
    n=20,
    best=True,
)

DatabaseError: Execution failed on sql '
        SELECT *
        FROM eval_results
        WHERE crystalbleu IS NOT NULL
        ORDER BY crystalbleu ASC
        LIMIT ?
    ': no such column: crystalbleu

In [5]:
worst["response"][1]["output"]

{'output': ''}

In [6]:
worst["test_case"][1]["vars"]["reference_image"]

'/images/datikz-test_row_00000170.png'

In [7]:
worst["grading_result"][0]

{'pass': False,
 'score': 0,
 'reason': 'CrystalBLEU failed: \'dict\' object has no attribute \'strip\'\nTraceback (most recent call last):\n  File "/app/assertions/tikz_crystalbleu.py", line 38, in get_assert\n    score = compute_crystalbleu_score(\n            ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/app/pf_utils/crystalbleu_metric.py", line 136, in compute_crystalbleu_score\n    generated_code = generated_code.strip()\n                     ^^^^^^^^^^^^^^^^^^^^\nAttributeError: \'dict\' object has no attribute \'strip\'\n',
 'namedScores': {'renderable': 0,
  'latex_errors': 0,
  'latex_warnings': 0,
  'latex_badboxes': 0,
  'image_ssim': 0,
  'ted_similarity': 0,
  'ted_distance': 999,
  'clip_similarity': 0,
  'siglip_similarity': 0,
  'lpips_similarity': 0,
  'lpips_distance': 999,
  'dists_distance': 0,
  'relative_length_error': 0,
  'dreamsim_similarity': 0,
  'ms-ssim': 0,
  'crystalbleu': 0},
 'namedScoreWeights': {'renderable': 1,
  'latex_errors': 1,
  'latex_warnings': 1,
  'l